# Phase 6 - Feature Engineering

Which customer characteristics best explain future churn?

## Why is this important?

Machine Learning models cannot learn directly from raw transaction records.

Customer behaviour must first be transformed into meaningful business features.

## Expected Output

Customer-level modelling dataset.

## Business Value

Creates a structured dataset suitable for churn prediction and customer risk analysis.

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

customer_df = pd.read_csv(
"/content/drive/MyDrive/Revenue_Intelligence_Project/Data/customer_analytics.csv"
)

rfm = pd.read_csv(
"/content/drive/MyDrive/Revenue_Intelligence_Project/Data/rfm_customers.csv"
)

customer_df["InvoiceDate"] = pd.to_datetime(
customer_df["InvoiceDate"]
)

Mounted at /content/drive


In [2]:
features = (
    customer_df[
        ["Customer ID"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

features.head()

,Customer ID
0,13085.0
1,13078.0
2,15362.0
3,18102.0
4,12682.0


In [3]:
features = features.merge(

    rfm[
        [
            "Customer ID",
            "Recency",
            "Frequency",
            "Monetary",
            "Segment"
        ]
    ],

    on="Customer ID",

    how="left"
)

features.head()

,Customer ID,Recency,Frequency,Monetary,Segment
0,13085.0,158,8,2433.28,Loyal Customers
1,13078.0,3,57,29018.06,Champions
2,15362.0,449,2,613.08,Lost Customers
3,18102.0,1,145,580987.04,Champions
4,12682.0,4,52,22950.22,Champions


In [4]:
customer_revenue = (

customer_df

.groupby("Customer ID")

["Revenue"]

.sum()

)

In [5]:
customer_orders = (

customer_df

.groupby("Customer ID")

["Invoice"]

.nunique()

)

In [6]:
aov = (

customer_revenue

/

customer_orders

).reset_index()

aov.columns = [

"Customer ID",

"AverageOrderValue"

]

In [7]:
features = features.merge(

aov,

on="Customer ID",

how="left"

)

In [8]:
lifetime = (

customer_df

.groupby("Customer ID")

["InvoiceDate"]

.agg(

["min","max"]

)

)

In [9]:
lifetime["CustomerLifetime"] = (

lifetime["max"]

-

lifetime["min"]

).dt.days

In [10]:
features = features.merge(

lifetime["CustomerLifetime"]

.reset_index(),

on="Customer ID",

how="left"

)

In [11]:
features.head()

,Customer ID,Recency,Frequency,Monetary,Segment,AverageOrderValue,CustomerLifetime
0,13085.0,158,8,2433.28,Loyal Customers,304.160000,581
1,13078.0,3,57,29018.06,Champions,509.088772,735
2,15362.0,449,2,613.08,Lost Customers,306.540000,290
3,18102.0,1,145,580987.04,Champions,4006.807172,738
4,12682.0,4,52,22950.22,Champions,441.350385,735


In [12]:
features.describe()

,Customer ID,Recency,Frequency,Monetary,AverageOrderValue,CustomerLifetime
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,15314.674205,201.457745,6.287196,2954.396237,384.975880,272.883183
std,1715.429759,209.474135,13.012879,14437.322635,1213.998407,258.815005
min,12346.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,13833.000000,26.000000,1.000000,341.900000,176.627143,0.000000
50%,15313.000000,96.000000,3.000000,865.600000,279.129091,220.000000
75%,16797.000000,380.000000,7.000000,2247.720000,414.895714,511.000000
max,18287.000000,739.000000,398.000000,580987.040000,84236.250000,738.000000


In [14]:
orders = pd.read_csv(
"/content/drive/MyDrive/Revenue_Intelligence_Project/Data/orders.csv"
)

order_intervals = pd.read_csv(
"/content/drive/MyDrive/Revenue_Intelligence_Project/Data/order_intervals.csv"
)

orders["InvoiceDate"] = pd.to_datetime(orders["InvoiceDate"])
orders["PreviousOrderDate"] = pd.to_datetime(orders["PreviousOrderDate"])

In [15]:
avg_gap = (
    order_intervals
    .groupby("Customer ID")["DaysBetweenOrders"]
    .mean()
    .reset_index()
)

avg_gap.columns = [
    "Customer ID",
    "AverageDaysBetweenOrders"
]

In [16]:
avg_gap = (
    order_intervals
    .groupby("Customer ID")["DaysBetweenOrders"]
    .mean()
    .reset_index()
)

avg_gap.columns = [
    "Customer ID",
    "AverageDaysBetweenOrders"
]

In [17]:
orders["InvoiceDate"] = pd.to_datetime(
    orders["InvoiceDate"]
)

orders["PreviousOrderDate"] = pd.to_datetime(
    orders["PreviousOrderDate"]
)

print("Orders Shape:", orders.shape)
print("Order Intervals Shape:", order_intervals.shape)

Orders Shape: (36975, 6)
Order Intervals Shape: (31094, 6)


In [18]:
# ==========================================
# Feature 6
# Average Days Between Orders
# ==========================================

avg_gap = (
    order_intervals
    .groupby("Customer ID")["DaysBetweenOrders"]
    .mean()
    .reset_index()
)

avg_gap.columns = [
    "Customer ID",
    "AverageDaysBetweenOrders"
]

avg_gap.head()

,Customer ID,AverageDaysBetweenOrders
0,12346.0,35.909091
1,12347.0,57.000000
2,12348.0,90.500000
3,12349.0,189.666667
4,12352.0,39.222222


In [19]:
features = features.merge(
    avg_gap,
    on="Customer ID",
    how="left"
)

features.head()

,Customer ID,Recency,Frequency,Monetary,Segment,AverageOrderValue,CustomerLifetime,AverageDaysBetweenOrders
0,13085.0,158,8,2433.28,Loyal Customers,304.160000,581,82.714286
1,13078.0,3,57,29018.06,Champions,509.088772,735,12.714286
2,15362.0,449,2,613.08,Lost Customers,306.540000,290,290.000000
3,18102.0,1,145,580987.04,Champions,4006.807172,738,4.868056
4,12682.0,4,52,22950.22,Champions,441.350385,735,13.901961


In [20]:
# ==========================================
# Feature 7
# Customer Age Category
# ==========================================

def customer_age(days):

    if days <= 90:
        return "New"

    elif days <= 365:
        return "Growing"

    else:
        return "Established"


features["CustomerAgeCategory"] = (
    features["CustomerLifetime"]
    .apply(customer_age)
)

features[
    [
        "CustomerLifetime",
        "CustomerAgeCategory"
    ]
].head()

,CustomerLifetime,CustomerAgeCategory
0,581,Established
1,735,Established
2,290,Growing
3,738,Established
4,735,Established


In [21]:
# ==========================================
# Feature 8
# Target Variable
# ==========================================

snapshot_date = (
    customer_df["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

last_purchase = (
    customer_df
    .groupby("Customer ID")["InvoiceDate"]
    .max()
    .reset_index()
)

last_purchase["DaysSinceLastPurchase"] = (
    snapshot_date
    - last_purchase["InvoiceDate"]
).dt.days

last_purchase["Churn"] = (
    last_purchase["DaysSinceLastPurchase"] > 90
).astype(int)

last_purchase.head()

,Customer ID,InvoiceDate,DaysSinceLastPurchase,Churn
0,12346.0,2011-01-18 10:01:00,326,1
1,12347.0,2011-12-07 15:52:00,2,0
2,12348.0,2011-09-25 13:13:00,75,0
3,12349.0,2011-11-21 09:51:00,19,0
4,12350.0,2011-02-02 16:01:00,310,1


In [22]:
features = features.merge(

    last_purchase[
        [
            "Customer ID",
            "Churn"
        ]
    ],

    on="Customer ID",

    how="left"
)

In [23]:
print(features.shape)

features.head()

(5881, 10)


,Customer ID,Recency,Frequency,Monetary,Segment,AverageOrderValue,CustomerLifetime,AverageDaysBetweenOrders,CustomerAgeCategory,Churn
0,13085.0,158,8,2433.28,Loyal Customers,304.160000,581,82.714286,Established,1
1,13078.0,3,57,29018.06,Champions,509.088772,735,12.714286,Established,0
2,15362.0,449,2,613.08,Lost Customers,306.540000,290,290.000000,Growing,1
3,18102.0,1,145,580987.04,Champions,4006.807172,738,4.868056,Established,0
4,12682.0,4,52,22950.22,Champions,441.350385,735,13.901961,Established,0


In [24]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer ID               5881 non-null   float64
 1   Recency                   5881 non-null   int64  
 2   Frequency                 5881 non-null   int64  
 3   Monetary                  5881 non-null   float64
 4   Segment                   5881 non-null   object 
 5   AverageOrderValue         5881 non-null   float64
 6   CustomerLifetime          5881 non-null   int64  
 7   AverageDaysBetweenOrders  4255 non-null   float64
 8   CustomerAgeCategory       5881 non-null   object 
 9   Churn                     5881 non-null   int64  
dtypes: float64(4), int64(4), object(2)
memory usage: 459.6+ KB


In [25]:
features["Churn"].value_counts()

,count
Churn,
1,2991
0,2890


In [26]:
# ==========================================
# Save Final Modeling Dataset
# ==========================================

features.to_csv(
    "/content/drive/MyDrive/Revenue_Intelligence_Project/Data/model_dataset.csv",
    index=False
)

print("Model Dataset Saved Successfully!")

Model Dataset Saved Successfully!


In [27]:
features["AverageDaysBetweenOrders"] = (
    features["AverageDaysBetweenOrders"]
    .fillna(0)
)

In [28]:
features.isnull().sum()

,0
Customer ID,0
Recency,0
Frequency,0
Monetary,0
Segment,0
AverageOrderValue,0
CustomerLifetime,0
AverageDaysBetweenOrders,0
CustomerAgeCategory,0
Churn,0
